In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 私榜重训:只需要改 w3b_model.py 里的一行
# ═══════════════════════════════════════════════════════════════════════════
#
#     TRAIN_START, TRAIN_END = "2019-08-01", "2024-12-31"
#                              └──────────┬──────────────┘
#                            改成私榜给的可见数据区间, 其余一律不动
# ═══════════════════════════════════════════════════════════════════════════

import os

from w3b_model import (
    MODEL_PATH, TABLE_KEY, BAR_TABLE,
    TRAIN_START, TRAIN_END, VAL_FRAC, SEEDS,
    train_and_save, predict,
)


def main(datasources, start_date, end_date):
    """平台入口: 加载 w3b_model.json, 在样本外测试区间打分, **不训练**。

    返回 ['date','instrument','score'] —— 已左连 PIT 全票池、缺失填当日截面均值,
    保证是完整面板(规则 A22),且每日缺失率会打印出来核对 40% 红线(规则 A16)。
    """
    return predict(datasources, start_date, end_date)


if __name__ == "__main__":
    from bigmodule import M

    datasources = {TABLE_KEY: BAR_TABLE}

    # ---------- 权重: 没有就训一次, 有就直接用 ----------
    if not os.path.exists(MODEL_PATH):
        print(f"未发现 {MODEL_PATH}, 开始训练 {len(SEEDS)} 个成员 seeds={SEEDS}")
        print(f"  可见区间 {TRAIN_START} ~ {TRAIN_END},末尾 {VAL_FRAC:.0%} 自动切作验证集(早停)")
        train_and_save(datasources)
    else:
        print(f"已存在 {MODEL_PATH}, 跳过训练, 直接推理")

    # ---------- 推理(本地用一小段模拟「平台注入的测试集区间」)----------
    # ⚠️ 这段落在 FIT 区间**内**(自动划分后大致就是验证集那一段) ⇒ 打印的评估**不是**干净的
    #    样本外, 只用来验证"推理链路跑得通、K 个成员都载入了、输出是完整面板"。
    #    真正的样本外成绩看平台榜。
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # ---------- 评估(分数经风格剔除后等价于每日单因子)----------
    # ★传 start/end: 不传的话评估器"回退到数据自身范围", 它自己标注那只建议本地调试用。
    #   传了之后官方评估窗口由我们指定, 与平台口径一致。
    #   (列名保持 score —— 官方模板就是 score, 评估器会自动重命名成 factor, 那是设计好的行为。)
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True,
                                     start_date=start_date, end_date=end_date)
